In [3]:
import spacy

nlp = spacy.load('en_core_web_sm')

print('spaCy ready!')

spaCy ready!


In [5]:
import re
from datetime import datetime

def extract_dates(text):
    """
    Extract dates in various formats
    Formats: MM/DD/YYYY, DD-MM-YYYY, Month DD, YYYY
    """
    patterns = [
        r'\d{1,2}/\d{1,2}/\d{4}',  # MM/DD/YYYY
        r'\d{1,2}-\d{1,2}-\d{4}',  # DD-MM-YYYY
        r'\b(?:Jan|Feb|Mar|Apr|May|Jun|Jul|Aug|Sep|Oct|Nov|Dec)[a-z]* \d{1,2},? \d{4}',
        r'\d{4}-\d{2}-\d{2}'  # ISO format
    ]

    dates = []
    for pattern in patterns:
        matches = re.findall(pattern, text)
        dates.extend(matches)

    return dates

# Test
text = "Invoice date: 03/15/2024. Due: March 30, 2024"
print(extract_dates(text))

['03/15/2024', 'March 30, 2024']


In [6]:
import re

def extract_amounts(text):
    """
    Extract currency amounts
    Handles: $1,250.50, 1250.50, $1250
    """
    pattern = r'\$?\d{1,3}(?:,\d{3})*(?:\.\d{2})?'
    amounts = re.findall(pattern, text)

    # Convert to float
    cleaned = []
    for amount in amounts:
        # Remove $ and ,
        clean = amount.replace('$', '').replace(',', '')
        cleaned.append(float(clean))

    return cleaned

# Test
text = "Total: $1,250.50. Tax: $125.05. Subtotal: 1125.45"
print(extract_amounts(text))

[1250.5, 125.05, 112.0, 5.45]


In [7]:
import re

def extract_invoice_number(text):
    """
    Extract invoice/order numbers
    Patterns: INV-2024-001, #12345, ORDER-ABC123
    """
    patterns = [
        r'INV-\d{4}-\d{3}',
        r'#\d{5,}',
        r'ORDER-[A-Z0-9]+',
        r'Invoice (?:Number|#):?\s*([A-Z0-9-]+)'
    ]

    for pattern in patterns:
        match = re.search(pattern, text, re.IGNORECASE)
        if match:
            # Return full match or first group
            return match.group(1) if match.groups() else match.group(0)

    return None

# Test
text = "Invoice Number: INV-2024-001"
print(extract_invoice_number(text))


INV-2024-001


 # PART 2: NAMED ENTITY RECOGNITION
 

In [8]:
import spacy

# Load model
nlp = spacy.load('en_core_web_sm')

# Sample invoice text
text = """
Invoice from Acme Corporation
123 Main Street, New York, NY 10001
Contact: John Smith (john@acme.com)
Date: March 15, 2024
Amount Due: $1,250.50
"""

# Process text
doc = nlp(text)

# Extract entities
print('Found entities:')
for ent in doc.ents:
    print(f'{ent.text:20} {ent.label_:15} {spacy.explain(ent.label_)}')

Found entities:
Acme Corporation     ORG             Companies, agencies, institutions, etc.
123                  CARDINAL        Numerals that do not fall under another type
Main Street          FAC             Buildings, airports, highways, bridges, etc.
New York             GPE             Countries, cities, states
10001                DATE            Absolute or relative dates or periods
John Smith           PERSON          People, including fictional
March 15, 2024       DATE            Absolute or relative dates or periods
1,250.50             MONEY           Monetary values, including unit


In [9]:
import spacy

# Load model (make sure this is already done once)
nlp = spacy.load('en_core_web_sm')

def extract_entities(text):
    """
    Extract and organize entities by type
    """
    doc = nlp(text)

    entities = {
        'persons': [],
        'organizations': [],
        'locations': [],
        'dates': [],
        'money': []
    }

    for ent in doc.ents:
        if ent.label_ == 'PERSON':
            entities['persons'].append(ent.text)
        elif ent.label_ == 'ORG':
            entities['organizations'].append(ent.text)
        elif ent.label_ in ['GPE', 'LOC']:
            entities['locations'].append(ent.text)
        elif ent.label_ == 'DATE':
            entities['dates'].append(ent.text)
        elif ent.label_ == 'MONEY':
            entities['money'].append(ent.text)

    return entities


# Test
text = """
Invoice from Acme Corporation
123 Main Street, New York, NY 10001
Contact: John Smith (john@acme.com)
Date: March 15, 2024
Amount Due: $1,250.50
"""

result = extract_entities(text)

for entity_type, values in result.items():
    print(f"{entity_type}: {values}")

persons: ['John Smith']
organizations: ['Acme Corporation']
locations: ['New York']
dates: ['10001', 'March 15, 2024']
money: ['1,250.50']


In [15]:
from spacy import displacy
import webbrowser

doc = nlp(text)

# Get HTML WITHOUT relying on return value
html_content = displacy.render(doc, style='ent')

# Force ensure it's string
html_content = str(html_content)

with open('entities.html', 'w', encoding='utf-8') as f:
    f.write(html_content)

webbrowser.open_new_tab('entities.html')

print("Visualization saved and opened in browser")

Visualization saved and opened in browser


# PART 3: COMPLETE EXTRACTION PIPELINE 

In [25]:
import json
import pytesseract
from PIL import Image

def process_invoice(image_path):
    """
    Complete pipeline: OCR → Extraction → JSON
    """

    # Step 1: OCR
    img = Image.open(image_path)
    text = pytesseract.image_to_string(img)

    # Step 2: Regex extraction
    invoice_data = {
        'invoice_number': extract_invoice_number(text),
        'dates': extract_dates(text),
        'amounts': extract_amounts(text)
    }

    # Step 3: NLP extraction
    entities = extract_entities(text)
    invoice_data.update(entities)

    # Step 4: Post-processing (FIXED INDENTATION)
    if invoice_data['amounts']:
        invoice_data['total_amount'] = max(invoice_data['amounts'])

    if invoice_data['dates']:
        invoice_data['invoice_date'] = invoice_data['dates'][0]

    return invoice_data




In [27]:
# Test

result = process_invoice("/kaggle/input/datasets/nayyabakhtar/my-ocr-dataset/receipt_6.jpg")

print(result)

{'invoice_number': None, 'dates': [], 'amounts': [4.0, 22.0, 202.0, 8.0, 100.0, 5.0, 199.66], 'persons': [], 'organizations': [], 'locations': [], 'money': ['1005', '199.66'], 'total_amount': 202.0}


In [28]:
import json

# Save to JSON file
output_file = 'extracted_data.json'

with open(output_file, 'w') as f:
    json.dump(result, f, indent=2)

print(f'Results saved to {output_file}')

Results saved to extracted_data.json
